### Cell 1 — Environment & Project Validation

This cell automatically detects the correct Maven project root by locating the pom.xml file, then identifies all src/main/java and src/test/java directories (including multi-module setups). It also counts the Java source files found and verifies that Maven is installed and accessible in the environment.

In [1]:
# ============================================================
# STEP 1 v3 — AUTO-DETECT MAVEN ROOT + ALL JAVA SOURCE ROOTS
#   - Finds the correct PROJECT_ROOT (folder containing pom.xml)
#   - Finds ALL */src/main/java and */src/test/java under it
#   - Counts Java files across all detected source roots
# ============================================================

import subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ---- Base workspace path (adjust only this if needed) ----
# Use the workspace root or a parent directory containing the Maven project.
# The old hard‑coded path pointed at a different repository; update it to
# match our current workspace. Using Path.cwd() would work if the notebook
# is launched from the repo root.
BASE_PATH = Path("/home/oussamaetta/Desktop/N7/Projects-S8/Projetweb").resolve()
# Alternatively: BASE_PATH = Path.cwd()

print(f"Scanning inside: {BASE_PATH}")

# ---- Auto-detect Maven project root (pom.xml) ----
pom_files = list(BASE_PATH.rglob("pom.xml"))
if not pom_files:
    raise Exception("❌ No pom.xml found inside BASE_PATH")

# For projects with multiple modules you could filter by directory name
# or other heuristics. Here there is only one pom under facade/ so just
# use the first match.
PROJECT_ROOT = pom_files[0].parent

print(f"\n✅ Maven project detected at:\n{PROJECT_ROOT}")

# ---- Detect ALL Java source roots (supports multi-module) ----
main_java_roots = sorted({p.parent for p in PROJECT_ROOT.rglob("src/main/java")})
test_java_roots = sorted({p.parent for p in PROJECT_ROOT.rglob("src/test/java")})

print("\n[Detected source roots]")
print("Main Java roots:")
for r in main_java_roots:
    print(" -", r.relative_to(PROJECT_ROOT))

print("\nTest Java roots:")
for r in test_java_roots:
    print(" -", r.relative_to(PROJECT_ROOT))

# ---- Count Java files across all main roots ----
main_java_files = []
for root in main_java_roots:
    main_java_files.extend(list(root.rglob("*.java")))

test_java_files = []
for root in test_java_roots:
    test_java_files.extend(list(root.rglob("*.java")))

print(f"\nTotal MAIN Java files detected: {len(main_java_files)}")
print(f"Total TEST Java files detected: {len(test_java_files)}")

print("\nSample MAIN Java files:")
for f in main_java_files[:8]:
    print("-", f.relative_to(PROJECT_ROOT))

# ---- Check Maven availability ----
print("\n[Checking Maven installation]")
result = subprocess.run(["mvn", "-v"], capture_output=True, text=True)
print(result.stdout.split("\n")[0])

Scanning inside: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb

✅ Maven project detected at:
/home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/facade

[Detected source roots]
Main Java roots:
 - src/main

Test Java roots:
 - src/test

Total MAIN Java files detected: 19
Total TEST Java files detected: 1

Sample MAIN Java files:
- src/main/java/n7/facade/Adherent.java
- src/main/java/n7/facade/Facade.java
- src/main/java/n7/facade/Discussion.java
- src/main/java/n7/facade/CommentRepository.java
- src/main/java/n7/facade/Recette.java
- src/main/java/n7/facade/Comment.java
- src/main/java/n7/facade/IngredientRepository.java
- src/main/java/n7/facade/AdherentRepository.java

[Checking Maven installation]
Apache Maven 3.6.3


###  Cell 2 -  Baseline Build (Ground Truth)

This cell runs a baseline mvn clean test build to establish the project’s ground truth before any AI modifications. It captures the exit code and recent logs, ensuring the project compiles and tests pass so you know any later failures are caused by AI-generated changes, not pre-existing issues.

In [2]:
# ==========================================
# STEP 2 — BASELINE BUILD (GROUND TRUTH)
# ==========================================

import subprocess
from datetime import datetime

print(f"Running baseline build in: {PROJECT_ROOT}")
print("Command: mvn clean test\n")

start = datetime.now()

result = subprocess.run(
    ["mvn", "clean", "test"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

end = datetime.now()
duration = (end - start).total_seconds()

print(f"Exit code: {result.returncode}")
print(f"Duration: {duration:.2f}s")

# Show last part of logs (most useful)
print("\n--- STDOUT (last 80 lines) ---")
stdout_lines = result.stdout.splitlines()
print("\n".join(stdout_lines[-80:]))

print("\n--- STDERR (last 80 lines) ---")
stderr_lines = result.stderr.splitlines()
print("\n".join(stderr_lines[-80:]))

# Simple status
if result.returncode == 0:
    print("\n✅ Baseline build PASSED. Safe to proceed to AI steps.")
else:
    print("\n❌ Baseline build FAILED. Fix project/env first before adding AI.")

Running baseline build in: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/facade
Command: mvn clean test

Exit code: 0
Duration: 13.04s

--- STDOUT (last 80 lines) ---
    alter table comment 
       add constraint FK95ohk1jcu4hdprq2ylk3wvek0 
       foreign key (auteur_id) 
       references adherent
Hibernate: 
    alter table comment 
       add constraint FKqd34j4uy105mc49dllp9q9pri 
       foreign key (recette_id) 
       references recette
Hibernate: 
    alter table discussion 
       add constraint FKqe1h9ds9ep26glgv679ca3nt1 
       foreign key (auteur_id) 
       references adherent
Hibernate: 
    alter table event 
       add constraint FK6hrjv8j9865q3a61xsqais4fw 
       foreign key (auteur_id) 
       references adherent
Hibernate: 
    alter table event_participants 
       add constraint FKpyorefbcs2xksthci6ll8t6o5 
       foreign key (adherent_id) 
       references adherent
Hibernate: 
    alter table event_participants 
       add constraint FK5232w1ta0icpkemgsxy

### Cell 3 - MINIMAL CONTEXT LOADER (NO RAG)

This cell builds a lightweight code context: it indexes all Java files, then selects a small relevant subset using keyword matching (seed files) and expands it by scanning those files for referenced class names to pull in likely dependencies. The goal is to feed the LLM only the most relevant files (plus pom.xml) instead of the whole repo, to save tokens and reduce noise.

its a 2-step filtering process: 
    - by key words
    - imported based / CamelCase

In [3]:
# ==========================================
# STEP 3 (v2) — MINIMAL CONTEXT + SMART PICKING
#   - Index java files
#   - Pick by keywords
#   - Expand context by scanning imports/usages (cheap heuristic)
# ==========================================

import re
from pathlib import Path

def read_text_file(path: Path, max_chars: int = 120_000) -> str:
    if not path.exists():
        return ""
    txt = path.read_text(encoding="utf-8", errors="replace")
    return txt[:max_chars]

# Base context
context = {}
context["pom.xml"] = read_text_file(PROJECT_ROOT / "pom.xml", max_chars=120_000) # Add pom.xml to context for better dependency understanding

# Index all main Java files (paths only)
java_index = []
for root in main_java_roots:
    java_index.extend(list(root.rglob("*.java")))
java_index = sorted(java_index)

# Build a map: ClassName -> path (simple: filename without .java)
class_to_path = {p.stem: p for p in java_index}

print(f"Indexed Java files (main): {len(java_index)}")

def pick_files_by_keywords(keywords, limit=8):
    kws = [k.lower() for k in keywords]
    matches = []
    for f in java_index:
        p = str(f).lower()
        if any(k in p for k in kws):
            matches.append(f)
    return matches[:limit]

def expand_by_references(seed_files, max_extra=6):
    """
    Cheap dependency expansion:
    - read seed files
    - find words that look like ClassNames (CamelCase)
    - if a ClassName exists in project, include its file too
    """
    extra = []
    seen = {sf for sf in seed_files}
    for sf in seed_files:
        txt = read_text_file(sf, max_chars=30_000)
        # find CamelCase tokens (very rough)
        candidates = set(re.findall(r"\b[A-Z][A-Za-z0-9_]{2,}\b", txt))
        for c in candidates:
            if c in class_to_path:
                p = class_to_path[c]
                if p not in seen:
                    extra.append(p)
                    seen.add(p)
                    if len(extra) >= max_extra:
                        return seed_files + extra
    return seed_files + extra

def expand_by_imports(seed_files, max_extra=6):
    """
    More precise dependency expansion:
    - Reads import statements from seed files
    - If imported class exists inside project, include its file
    """
    extra = []
    seen = set(seed_files)

    for sf in seed_files:
        txt = read_text_file(sf, max_chars=40_000)

        # Extract import lines
        imports = re.findall(r"import\s+([\w\.]+);", txt)

        for imp in imports:
            class_name = imp.split(".")[-1]  # get final class name
            if class_name in class_to_path:
                p = class_to_path[class_name]
                if p not in seen:
                    extra.append(p)
                    seen.add(p)
                    if len(extra) >= max_extra:
                        return seed_files + extra

    return seed_files + extra

# Example (adjust keywords to your project domain before running)
# This repository deals with adherents, events, recettes, etc.; pick some
# representative terms so the context loader finds relevant controllers.
seed = pick_files_by_keywords(["adherent", "event", "recette"], limit=5)
# picked_paths = expand_by_references(seed, max_extra=6)
picked_paths = expand_by_imports(seed, max_extra=6)


print("\nPicked files (seed + expanded):")
for f in picked_paths:
    print("-", f.relative_to(PROJECT_ROOT))

Indexed Java files (main): 19

Picked files (seed + expanded):
- src/main/java/n7/facade/Adherent.java
- src/main/java/n7/facade/AdherentController.java
- src/main/java/n7/facade/AdherentRepository.java
- src/main/java/n7/facade/Event.java
- src/main/java/n7/facade/EventRepository.java


### Cell 4 — Plan-only LLM (no code writing)

This cell asks the LLM to analyze the selected project context and produce a structured implementation plan (JSON only) for a requested feature — without generating any code yet.

It forces the model to:

- Respect existing APIs (no hallucinated methods),

- Specify which files to create/modify,

- Define tests, API contract, risks, and acceptance checks  (before moving to code generation)

In [ ]:
# ==========================================
# STEP 4 (v2) — PLAN-ONLY LLM (STRICT, NO HALLUCINATED APIS)
# ==========================================

import os
import json
import re
# Robust imports: prefer provider-specific packages but fall back to main langchain
try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
except Exception:
    try:
        # Newer langchain: chat_models provides ChatOpenAI wrapper
        from langchain.chat_models import ChatOpenAI
    except Exception:
        ChatOpenAI = None
    try:
        # Prompt templates live under langchain.prompts in many installs
        from langchain.prompts import ChatPromptTemplate
    except Exception:
        ChatPromptTemplate = None
    if ChatOpenAI is None or ChatPromptTemplate is None:
        print("⚠️ Warning: LangChain import fallbacks unavailable. Ensure 'langchain' or 'langchain_openai' is installed.")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
# MODEL_NAME = os.getenv("MODEL_NAME", "anthropic/claude-3.5-sonnet")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

if not OPENROUTER_API_KEY:
    raise Exception("❌ Missing OPENROUTER_API_KEY in your .env")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.2,
    max_tokens=650,   # keep low for budget
)

# written by dev
#FEATURE_REQUEST = """
#Add a REST endpoint to fetch a student's progress summary.
#Return: studentId, totalPoints, level, badgesCount.
#Include basic validation and proper HTTP status codes.
#""" 
 
# Detect base Java package from existing source files (e.g. n7.facade)
def detect_base_package(project_root):
    java_main = project_root / "src/main/java"
    candidates = sorted(java_main.rglob("*.java")) if java_main.exists() else []
    for jf in candidates:
        txt = jf.read_text(encoding="utf-8", errors="replace")
        m = re.search(r"^\s*package\s+([a-zA-Z0-9_.]+)\s*;", txt, flags=re.MULTILINE)
        if m:
            return m.group(1)
    return None

BASE_PACKAGE = detect_base_package(PROJECT_ROOT)
if not BASE_PACKAGE:
    raise Exception("❌ Could not detect base Java package under src/main/java")
BASE_PACKAGE_PATH = BASE_PACKAGE.replace(".", "/")
ALLOWED_MAIN_TARGET = f"src/main/java/{BASE_PACKAGE_PATH}/AiPocController.java"
ALLOWED_TEST_TARGET = f"src/test/java/{BASE_PACKAGE_PATH}/AiPocControllerWebMvcTest.java"
print(f"Using package: {BASE_PACKAGE}")

# written by dev
FEATURE_REQUEST = f"""
Add a minimal proof-of-concept REST endpoint to validate the AI pipeline.

API:
- Method: GET
- Path: /ai/poc
- Success response (200) JSON:
  {
    "status": "ok",
    "timestamp": "<ISO-8601 string>"
  }

Validation:
- If a query param `bad=true` is provided, return HTTP 400 with JSON:
  { "error": "bad request" }

HARD CONSTRAINTS (must be enforced):
- Do NOT access the database.
- Do NOT create or modify any Service/Repository/DAO classes.
- Do NOT modify existing project files (files_to_modify MUST be []).
- Only allowed changes: create EXACTLY these two new files:
  1) {ALLOWED_MAIN_TARGET}
  2) {ALLOWED_TEST_TARGET}

Testing:
- Add a WebMvcTest (MockMvc) that checks:
  - GET /ai/poc returns 200 and JSON has keys status,timestamp (status == "ok")
  - GET /ai/poc?bad=true returns 400 and JSON contains key error

Goal:
- Minimal change. No integration with existing features. Just prove the pipeline works end-to-end.
"""

# Load only picked files (from Cell 3 v2)
picked_files = {}
for p in picked_paths[:10]:
    picked_files[str(p.relative_to(PROJECT_ROOT))] = read_text_file(p, max_chars=6000)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior Spring Boot engineer. Produce ONLY valid JSON. No markdown."),
    ("human",
     """You must plan changes for a Java/Spring Boot Maven project.

CRITICAL RULE:
- You may NOT invent calls to methods/classes that are not present in the provided context.
- If needed methods do not exist, you MUST plan to add them (and plan tests).

Context:
- pom.xml (partial)
{pom}

Picked project code context:
{picked_context}

FEATURE REQUEST:
{feature_request}

Return STRICT JSON with schema:
{{
  "summary": "...",
  "files_to_modify": ["..."],
  "files_to_create": ["..."],
  "tests_to_add": [
    {{"type":"unit|webmvc|integration", "target":"...", "notes":"..."}}
  ],
  "api_contract": {{
    "method":"GET",
    "path":"/...",
    "responses":[{{"status":200,"body_example":{{}}}},{{"status":400,"body_example":{{}}}}]
  }},
  "acceptance_checks": ["mvn test passes", "..."],
  "risks": ["..."]
}}"""
    )
])

msg = prompt.format_messages(
    pom=context["pom.xml"][:1200],
    picked_context=json.dumps({k: v[:2500] for k, v in picked_files.items()}, ensure_ascii=False, indent=2),
    feature_request=FEATURE_REQUEST.strip()
)

resp = llm.invoke(msg).content

try:
    plan = json.loads(resp)

    # Enforce/normalize target paths to this project package
    def normalize_target_path(p):
        n = str(p).replace("\\", "/").strip()
        if n.endswith("/AiPocController.java"):
            return ALLOWED_MAIN_TARGET
        if n.endswith("/AiPocControllerWebMvcTest.java"):
            return ALLOWED_TEST_TARGET
        return n

    plan["files_to_create"] = [normalize_target_path(p) for p in (plan.get("files_to_create") or [])]
    plan["files_to_modify"] = [normalize_target_path(p) for p in (plan.get("files_to_modify") or [])]

    allowed = {ALLOWED_MAIN_TARGET, ALLOWED_TEST_TARGET}
    bad_targets = [
        p for p in (plan["files_to_create"] + plan["files_to_modify"])
        if p not in allowed
    ]
    if bad_targets:
        raise Exception("❌ Plan contains disallowed paths: " + json.dumps(bad_targets, ensure_ascii=False))

    # Keep tests_to_add target aligned too
    for t in (plan.get("tests_to_add") or []):
        if isinstance(t, dict) and "target" in t:
            t["target"] = normalize_target_path(t["target"])

    print("✅ Plan JSON parsed.\n")
    print(json.dumps(plan, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("❌ Invalid JSON. Raw output:\n")
    print(resp)


✅ Plan JSON parsed.

{
  "summary": "Add a minimal proof-of-concept REST endpoint to validate the AI pipeline.",
  "files_to_modify": [],
  "files_to_create": [
    "src/main/java/n7/facade/AiPocController.java",
    "src/test/java/n7/facade/AiPocControllerWebMvcTest.java"
  ],
  "tests_to_add": [
    {
      "type": "webmvc",
      "target": "src/test/java/n7/facade/AiPocControllerWebMvcTest.java",
      "notes": "Test GET /ai/poc for successful response and bad request handling."
    }
  ],
  "api_contract": {
    "method": "GET",
    "path": "/ai/poc",
    "responses": [
      {
        "status": 200,
        "body_example": {
          "status": "ok",
          "timestamp": "<ISO-8601 string>"
        }
      },
      {
        "status": 400,
        "body_example": {
          "error": "bad request"
        }
      }
    ]
  },
  "acceptance_checks": [
    "mvn test passes"
  ],
  "risks": [
    "No existing integration with other features, isolated endpoint."
  ]
}


### Cell 5 — Generate Code + Tests (Dry Run, No Saving)

On demande au LLM de décider quels fichiers créer ou modifier (patchset).

Puis on appelle le LLM une fois par fichier pour générer son contenu complet.

On applique des règles de sécurité (pas de markdown, accolades équilibrées, pas de texte parasite).

On stocke les fichiers générés en mémoire (generated_files) sans encore les écrire sur disque.

In [6]:
# ==========================================
# STEP 5 — EXECUTE PLAN (STRICT PIPELINE)
#   - Uses ONLY `plan` (Cell 4)
#   - Uses ONLY `picked_files` (Cell 3)
#   - Generates tests first
#   - One LLM call per file
#   - Strict safety rules
#   - Stores in memory (generated_files)
# ==========================================

import os, json, re
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ---- Preconditions ----
if "PROJECT_ROOT" not in globals():
    raise Exception("❌ Run Cell 1 first.")
if "picked_files" not in globals() or not picked_files:
    raise Exception("❌ Run Cell 3 first (picked_files missing).")
if "plan" not in globals() or not isinstance(plan, dict):
    raise Exception("❌ Run Cell 4 first (plan missing).")

if "generated_files" not in globals():
    generated_files = {}

# ---- Model ----
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

if not OPENROUTER_API_KEY:
    raise Exception("❌ Missing OPENROUTER_API_KEY.")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.1,
    max_tokens=900,
)

# ---- Safety helpers ----
def strip_fences(s: str) -> str:
    s = s.lstrip()
    s = re.sub(r"```[a-zA-Z0-9_-]*\s*\n", "", s)
    s = s.replace("\n```", "\n")
    return s.strip() + "\n"

def java_safety(rel_path: str, content: str):
    head = content.lstrip()[:200].lower()

    if head.startswith(("file:", "here", "sure", "i'll", "i will")):
        raise Exception(f"❌ {rel_path}: wrapper text detected.")

    if "`" in content[:400] or "```" in content[:400]:
        raise Exception(f"❌ {rel_path}: markdown detected.")

    if rel_path.endswith(".java"):
        if content.count("{") != content.count("}"):
            raise Exception(f"❌ {rel_path}: unbalanced braces (truncated output).")

        h = content.lstrip()[:200]
        if not (
            h.startswith("package ")
            or h.startswith("import ")
            or h.startswith("@")
            or h.startswith("public ")
            or h.startswith("/*")
        ):
            raise Exception(f"❌ {rel_path}: suspicious Java header.")

# ---- Context from Cell 3 ONLY ----
repo_ctx = {
    k: (v or "")[:2000]   # small trim for token safety
    for k, v in list(picked_files.items())[:10]
}

# ---- Targets from plan ----
files_to_create = plan.get("files_to_create", []) or []
files_to_modify = plan.get("files_to_modify", []) or []

# Normalize/validate paths against detected package
if "BASE_PACKAGE_PATH" not in globals() or not BASE_PACKAGE_PATH:
    raise Exception("❌ BASE_PACKAGE_PATH missing. Run Cell 4 first.")

allowed_main = f"src/main/java/{BASE_PACKAGE_PATH}/AiPocController.java"
allowed_test = f"src/test/java/{BASE_PACKAGE_PATH}/AiPocControllerWebMvcTest.java"
allowed_targets = {allowed_main, allowed_test}

def normalize_target_path(p):
    n = str(p).replace("\\", "/").strip()
    if n.endswith("/AiPocController.java"):
        return allowed_main
    if n.endswith("/AiPocControllerWebMvcTest.java"):
        return allowed_test
    return n

files_to_create = [normalize_target_path(p) for p in files_to_create]
files_to_modify = [normalize_target_path(p) for p in files_to_modify]
bad = [p for p in (files_to_create + files_to_modify) if p not in allowed_targets]
if bad:
    raise Exception("❌ Disallowed target path(s): " + json.dumps(bad, ensure_ascii=False))

targets = []
for p in files_to_create:
    targets.append(("create", p))
for p in files_to_modify:
    targets.append(("modify", p))

if not targets:
    raise Exception("❌ Plan contains no files to create/modify.")

# ---- Tests first ----
def is_test(p):
    return p.replace("\\", "/").startswith("src/test/java/")

targets = sorted(targets, key=lambda t: (0 if is_test(t[1]) else 1, t[1]))
targets = targets[:5]  # guardrail

print("Targets (tests first):")
for action, p in targets:
    print("-", action, p)

# ---- Prompt template ----
file_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior Spring Boot engineer. Output ONLY raw Java file content. No markdown. No explanations."),
    ("human",
     """Generate the FULL content for EXACTLY ONE file.

PATH: {path}
ACTION: {action}

Rules:
- Output ONLY the file content.
- No markdown fences.
- No commentary.
- Keep changes minimal and consistent with existing project style.
- For tests: prefer WebMvcTest(MockMvc).
- For controllers: avoid try/catch.

Repo context:
{repo_ctx}

Existing file content:
{existing}

Plan JSON:
{plan_json}
""")
])

new_generated = {}

for action, path in targets:
    abs_path = PROJECT_ROOT / path
    existing = ""
    if abs_path.exists():
        existing = abs_path.read_text(encoding="utf-8", errors="replace")[:6000]

    msg = file_prompt.format_messages(
        path=path,
        action=action,
        repo_ctx=json.dumps(repo_ctx, ensure_ascii=False, indent=2),
        existing=existing,
        plan_json=json.dumps(plan, ensure_ascii=False, indent=2),
    )

    raw = llm.invoke(msg).content
    content = strip_fences(raw)

    java_safety(path, content)

    new_generated[path] = content
    print(f"✅ Prepared {path} (chars={len(content)})")

generated_files.update(new_generated)

print("\nDone. Files stored in memory (generated_files).")
print("Next: run Cell 6.")

Targets (tests first):
- create src/test/java/n7/facade/AiPocControllerWebMvcTest.java
- create src/main/java/n7/facade/AiPocController.java
✅ Prepared src/test/java/n7/facade/AiPocControllerWebMvcTest.java (chars=1290)
✅ Prepared src/main/java/n7/facade/AiPocController.java (chars=728)

Done. Files stored in memory (generated_files).
Next: run Cell 6.


### Cell 6 - SAFE APPLY PATCH + MVN TEST + ROLLBACK

In [9]:
# ==========================================
# STEP 6 (v3) — APPLY + MVN TEST + SAVE ARTIFACTS + ROLLBACK
#   - Safety checks before writing
#   - Backup overwritten files
#   - Write generated files
#   - Run mvn test
#   - If failure: save artifacts (generated files + logs + meta) then rollback
# ==========================================

import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

if "generated_files" not in globals() or len(generated_files) == 0:
    raise Exception("❌ No generated_files. Run Cell 5 first.")
if "BASE_PACKAGE_PATH" not in globals() or not BASE_PACKAGE_PATH:
    raise Exception("❌ BASE_PACKAGE_PATH missing. Run Cell 4 first.")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR = PROJECT_ROOT / ".ai_backups" / RUN_ID
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PROJECT_ROOT / ".ai_artifacts" / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

overwritten, created = [], []

# -------------------------------
# SAFETY CHECKS (before writing)
# -------------------------------
for rel_path, content in generated_files.items():
    # 0) Refuse unexpected paths outside this project's package
    norm = str(rel_path).replace("\\", "/")
    allowed_prefix_main = f"src/main/java/{BASE_PACKAGE_PATH}/"
    allowed_prefix_test = f"src/test/java/{BASE_PACKAGE_PATH}/"
    if not (norm.startswith(allowed_prefix_main) or norm.startswith(allowed_prefix_test)):
        raise Exception(f"❌ Refusing to write {rel_path}: outside allowed package path {BASE_PACKAGE_PATH}")

    # 1) Refuse if markdown/code fences leaked into code
    if "`" in content[:300]:
        raise Exception(
            f"❌ Refusing to write {rel_path}: contains backticks/code fences near the top. "
            "Fix Cell 5 parsing/strip fences."
        )

    # 2) Refuse if it doesn't look like a Java file header (for .java files)
    if rel_path.endswith(".java"):
        head = content.lstrip()[:200]
        if not (head.startswith("package ") or head.startswith("import ") or head.startswith("public ") or head.startswith("@")):
            raise Exception(
                f"❌ Refusing to write {rel_path}: file header looks suspicious (no package/import/public/@). "
                "Model likely added explanation text."
            )

# -------------------------------
# APPLY PATCH (with backups)
# -------------------------------
for rel_path, content in generated_files.items():
    target = PROJECT_ROOT / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        backup_path = BACKUP_DIR / rel_path
        backup_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, backup_path)
        overwritten.append(rel_path)
    else:
        created.append(rel_path)

    target.write_text(content, encoding="utf-8")

print(f"Backup directory: {BACKUP_DIR}")
print("Artifacts directory: ", ARTIFACT_DIR)
print("Overwritten:", overwritten if overwritten else "None")
print("Created:", created if created else "None")

# -------------------------------
# RUN TESTS
# -------------------------------
result = subprocess.run(
    ["mvn", "test"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

last_mvn_stdout = result.stdout
last_mvn_stderr = result.stderr
last_mvn_code = result.returncode

print("\nExit code:", last_mvn_code)
print("\n--- STDOUT (last 60 lines) ---")
print("\n".join(last_mvn_stdout.splitlines()[-60:]))
print("\n--- STDERR (last 60 lines) ---")
print("\n".join(last_mvn_stderr.splitlines()[-60:]))

# -------------------------------
# SAVE ARTIFACTS ALWAYS (useful even on success)
# -------------------------------
meta = {
    "run_id": RUN_ID,
    "overwritten": overwritten,
    "created": created,
    "mvn_exit_code": last_mvn_code,
}
(ARTIFACT_DIR / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "mvn_stdout.txt").write_text(last_mvn_stdout, encoding="utf-8")
(ARTIFACT_DIR / "mvn_stderr.txt").write_text(last_mvn_stderr, encoding="utf-8")

gen_dump_dir = ARTIFACT_DIR / "generated_files"
gen_dump_dir.mkdir(parents=True, exist_ok=True)

for rel_path, content in generated_files.items():
    out_path = gen_dump_dir / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(content, encoding="utf-8")

print(f"\n🧾 Saved attempt artifacts to: {ARTIFACT_DIR}")

# -------------------------------
# ROLLBACK ON FAILURE
# -------------------------------
if last_mvn_code != 0:
    print("\n❌ mvn test failed. Rolling back...")

    for rel_path in overwritten:
        backup_file = BACKUP_DIR / rel_path
        target = PROJECT_ROOT / rel_path
        if backup_file.exists():
            shutil.copy2(backup_file, target)

    for rel_path in created:
        target = PROJECT_ROOT / rel_path
        if target.exists():
            target.unlink()

    print("✅ Rollback done. Repo restored. Artifacts preserved in .ai_artifacts.")
else:
    print("\n✅ mvn test PASSED. Patch kept.")
    print("Artifacts preserved in .ai_artifacts (useful for reporting).")

Backup directory: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/facade/.ai_backups/20260303_232217
Artifacts directory:  /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/facade/.ai_artifacts/20260303_232217
Overwritten: None
Created: ['src/test/java/n7/facade/AiPocControllerWebMvcTest.java', 'src/main/java/n7/facade/AiPocController.java']

Exit code: 1

--- STDOUT (last 60 lines) ---
Hibernate: 
    alter table message 
       add constraint FK3nr1t44itdoagqs1w2bo21ho7 
       foreign key (discussion_id) 
       references discussion
Hibernate: 
    alter table recette 
       add constraint FKk7knhixq3ytr6d6qlhx0xu91j 
       foreign key (auteur_id) 
       references adherent
Hibernate: 
    alter table recette_categories 
       add constraint FKr1u637vqprjempxfg0wdggesl 
       foreign key (recette_id_rec) 
       references recette
Hibernate: 
    alter table recette_etapes 
       add constraint FK60lhnjbk43l4ci1901ps7de36 
       foreign key (recette_id_rec) 
       refer

In Cell 7 (self-repair) we feed the LLM three main things:

1️⃣ The failure signal (most important)

From Cell 6:

last_mvn_stdout

last_mvn_stderr

Reduced to failure_tail (last ~180–220 lines)

👉 This tells the model what broke (404 vs 400, missing method, wrong return type, etc.).

In [8]:
# ==========================================
# STEP 7 (compact) — SELF-REPAIR (uses ONLY prior cells)
# Requires:
#   - PROJECT_ROOT (Cell 1)
#   - picked_files (Cell 4 context built from Cell 3)
#   - last_mvn_code/stdout/stderr (Cell 6)
# Produces:
#   - updates generated_files (in-memory) so you can rerun Cell 6
# ==========================================

import os, json, re
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ---- prereqs ----
if "PROJECT_ROOT" not in globals():
    raise Exception("❌ PROJECT_ROOT missing. Run Cell 1.")
if "BASE_PACKAGE_PATH" not in globals() or not BASE_PACKAGE_PATH:
    raise Exception("❌ BASE_PACKAGE_PATH missing. Run Cell 4.")
if "picked_files" not in globals() or not isinstance(picked_files, dict) or not picked_files:
    raise Exception("❌ picked_files missing/empty. Run Cell 4 (it builds picked_files from Cell 3).")
if "last_mvn_code" not in globals():
    raise Exception("❌ last_mvn_code missing. Run Cell 6 first.")
if last_mvn_code == 0:
    print("✅ Build already passing. Nothing to repair.")
    raise SystemExit
if "generated_files" not in globals():
    generated_files = {}

# ---- model ----
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")  # default to gpt-4o-mini
if not OPENROUTER_API_KEY:
    raise Exception("❌ Missing OPENROUTER_API_KEY in .env")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.1,
    max_tokens=900,
)

# ---- small helpers ----
def strip_fences(s: str) -> str:
    s = s.lstrip()
    s = re.sub(r"```[a-zA-Z0-9_-]*\s*\n", "", s)
    s = s.replace("\n```", "\n")
    return s.strip()

def safe_json_load(s: str):
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

def java_sanity(rel_path: str, content: str):
    if "```" in content[:300] or "`" in content[:300]:
        raise Exception(f"❌ Markdown leaked into {rel_path}")
    if rel_path.endswith(".java") and content.count("{") != content.count("}"):
        raise Exception(f"❌ {rel_path}: unbalanced braces (likely truncated)")

def insert_before_last_brace(existing: str, snippet: str) -> str:
    idx = existing.rfind("}")
    if idx == -1:
        raise Exception("❌ Cannot patch: no closing brace found.")
    snippet = snippet.strip()
    if snippet.count("{") != snippet.count("}"):
        raise Exception("❌ Patch snippet truncated (unbalanced braces).")
    out = existing[:idx].rstrip() + "\n\n" + snippet + "\n\n" + existing[idx:]
    if out.count("{") != out.count("}"):
        raise Exception("❌ Patch insertion broke brace balance.")
    return out

def is_big_risk(path: str, existing_len: int) -> bool:
    return path.endswith("ProgressTracker.java") or existing_len > 9000

# ---- inputs from Cell 6 ----
last_stdout = globals().get("last_mvn_stdout", "") or ""
last_stderr = globals().get("last_mvn_stderr", "") or ""
failure_tail = "\n".join((last_stdout + "\n" + last_stderr).splitlines()[-220:])

# ---- context from prior cells ONLY (truncate for budget) ----
# include generated_files too so that the LLM can see and repair newly created/modified files
all_ctx = {**picked_files, **generated_files}  # generated_files may be empty initially
base_ctx = {k: v[:1200] for k, v in all_ctx.items()}
plan_json = json.dumps(plan, ensure_ascii=False, indent=2) if "plan" in globals() else "{}"

# ==========================================
# A) Ask LLM for patchset JSON (no code)
# ==========================================
patchset_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY valid JSON. No markdown. No explanations. No code blocks."),
    ("human",
     """mvn test failed. Propose a minimal repair patchset.

Failure tail:\n{failure_tail}\n
Context excerpts:\n{base_ctx}\n
Plan (read-only):\n{plan_json}\n
Return JSON EXACTLY:
{{
  "modify":[{{"path":"...","reason":"...","mode":"full|patch"}}],
  "create":[{{"path":"...","reason":"...","mode":"full"}}]
}}

Rules:
- Max 3 files total.
- Only .java under src/main/java or src/test/java.
- mode="patch" means later output ONLY a small snippet to insert (e.g., one method).

HARD CONSTRAINTS:

- You may ONLY modify files that were modified or created in the previous attempt.
- You MUST NOT modify any pre-existing core project files.
- Especially DO NOT modify:
  - ProgressController
  - ProgressTracker
  - PostgreSQLCommunication
  - Any file under Services/ unless it was newly created in the last attempt.

- If you think another file must change, DO NOT propose it.
  Instead, adjust ONLY the files already modified/created previously.

- Prefer fixing tests or newly created controllers rather than touching core logic.
"""
    )
])

raw = llm.invoke(patchset_prompt.format_messages(
    failure_tail=failure_tail,
    base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2),
    plan_json=plan_json
)).content

patchset = safe_json_load(strip_fences(raw))
if patchset is None:
    raise Exception("❌ Could not parse patchset JSON. Raw:\n" + raw[:800])

targets = []
for it in (patchset.get("modify") or []):
    targets.append(("modify", it["path"], it.get("reason",""), it.get("mode","full")))
for it in (patchset.get("create") or []):
    targets.append(("create", it["path"], it.get("reason",""), "full"))

allowed_main = f"src/main/java/{BASE_PACKAGE_PATH}/AiPocController.java"
allowed_test = f"src/test/java/{BASE_PACKAGE_PATH}/AiPocControllerWebMvcTest.java"
allowed_targets = {allowed_main, allowed_test}

def normalize_target_path(p):
    n = str(p).replace("\\", "/").strip()
    if n.endswith("/AiPocController.java"):
        return allowed_main
    if n.endswith("/AiPocControllerWebMvcTest.java"):
        return allowed_test
    return n

targets = [(a, normalize_target_path(p), r, m) for (a, p, r, m) in targets]
bad_targets = [p for (a, p, r, m) in targets if p not in allowed_targets]
if bad_targets:
    raise Exception("❌ Repair patchset contains disallowed paths: " + json.dumps(bad_targets, ensure_ascii=False))
targets = targets[:3]
if not targets:
    raise Exception("❌ Patchset empty.")

print("✅ Patchset:")
for a,p,r,m in targets:
    print(f"- {a} ({m}): {p} | {r}")

# ==========================================
# B) Generate per-file (full or patch)
# ==========================================
full_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY raw file content. No markdown. No explanations."),
    ("human",
     """Generate FULL content for ONE file: {path}

Action: {action}
Reason: {reason}

Failure tail:\n{failure_tail}\n
Context excerpts:\n{base_ctx}\n
Plan (read-only):\n{plan_json}\n
Existing file (empty if new):\n{existing}\n
Important instructions for JSON responses and maps:
- When returning JSON maps in controller responses, prefer Map<String, Object> for body maps so values can be heterogeneous.
- Ensure any local map variable types match the declared ResponseEntity generic. Avoid using Map<String, String> when the method returns ResponseEntity<Map<String, Object>>.
"""
    )
])

patch_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY a Java snippet to INSERT into the existing class. No markdown. No explanations."),
    ("human",
     """Generate ONLY the MINIMAL Java snippet to fix: {path}

Action: {action}
Reason: {reason}

Rules:
- Output ONLY the snippet (no package/import/class header).
- No markdown fences, no extra text.

HARD CONSTRAINTS:

- You may ONLY modify files that were modified or created in the previous attempt.
- You MUST NOT modify any pre-existing core project files.
- Especially DO NOT modify:
  - ProgressController
  - ProgressTracker
  - PostgreSQLCommunication
  - Any file under Services/ unless it was newly created in the last attempt.

- If you think another file must change, DO NOT propose it.
  Instead, adjust ONLY the files already modified/created previously.

- Prefer fixing tests or newly created controllers rather than touching core logic.

Important:
- If the fix involves returning or building JSON response maps, use Map<String, Object> for the body map and ensure declared variable types match the controller's ResponseEntity generic.

Failure tail:\n{failure_tail}\n
Existing file (reference):\n{existing}\n"""
    )
])

new_generated = {}

for action, path, reason, mode in targets:
    abs_path = PROJECT_ROOT / path
    existing_full = abs_path.read_text(encoding="utf-8", errors="replace") if abs_path.exists() else ""
    mode_eff = "patch" if (action == "modify" and (mode == "patch" or is_big_risk(path, len(existing_full))) and abs_path.exists()) else "full"

    if mode_eff == "full":
        msg = full_prompt.format_messages(
            path=path,
            action=action,
            reason=reason,
            failure_tail=failure_tail,
            base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2),
            plan_json=plan_json,
            existing=existing_full[:8000]
        )
        content = strip_fences(llm.invoke(msg).content)
        java_sanity(path, content)
        new_generated[path] = content + ("\n" if not content.endswith("\n") else "")
        print(f"✅ FULL: {path}")

    else:
        # patch mode: file must exist (we already ensured that above), but double-check
        if not abs_path.exists():
            print(f"⚠️ File {path} missing; falling back to full generation instead of patch.")
            # regenerate using full logic
            msg = full_prompt.format_messages(
                path=path,
                action=action,
                reason=reason,
                failure_tail=failure_tail,
                base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2),
                plan_json=plan_json,
                existing=existing_full[:8000]
            )
            content = strip_fences(llm.invoke(msg).content)
            java_sanity(path, content)
            new_generated[path] = content + ("\n" if not content.endswith("\n") else "")
            print(f"✅ FULL (fallback): {path}")
        else:
            msg = patch_prompt.format_messages(
                path=path,
                action=action,
                reason=reason,
                failure_tail=failure_tail,
                existing=existing_full[:8000]
            )
            snippet = strip_fences(llm.invoke(msg).content)
            if "```" in snippet[:200] or "`" in snippet[:200]:
                raise Exception(f"❌ Markdown in patch snippet for {path}")
            patched = insert_before_last_brace(existing_full, snippet)
            java_sanity(path, patched)
            new_generated[path] = patched + ("\n" if not patched.endswith("\n") else "")
            print(f"✅ PATCH: {path}")

# Quick auto-fix for a common generic mismatch: if the generated file
# declares a ResponseEntity<Map<String, Object>> but uses Map<String, String>
# for the response body, replace the local map type to Map<String, Object>.
for p, content in list(new_generated.items()):
    try:
        if 'ResponseEntity<Map<String, Object>>' in content and 'Map<String, String>' in content:
            fixed = re.sub(r'Map\s*<\s*String\s*,\s*String\s*>', 'Map<String, Object>', content)
            new_generated[p] = fixed
            print(f"🔧 Auto-fixed generics in: {p}")
    except Exception:
        pass

generated_files.update(new_generated)

print("\nDone. Added/updated in generated_files:")
for k in new_generated.keys():
    print("-", k)

print("\nNow rerun Cell 6.")

✅ Patchset:
- modify (patch): src/test/java/n7/facade/AiPocControllerWebMvcTest.java | Add missing Spring Boot configuration to the test class.
✅ FULL: src/test/java/n7/facade/AiPocControllerWebMvcTest.java

Done. Added/updated in generated_files:
- src/test/java/n7/facade/AiPocControllerWebMvcTest.java

Now rerun Cell 6.
